# Phase 4 — Consolidated Cross-Modality Evaluation

**MSc dissertation — cross-modality diabetic retinopathy (EyePACS colour fundus → OLIVES near-IR fundus).**

This notebook is a **thin orchestrator**; all logic lives in
`src/analysis/phase4_report.py`. It performs **no new computation** — it reads
the saved Phase 3a/4a/4b/4c JSONs + reports and assembles one coherent, honest
write-up (`PHASE4_RESULTS.md`) telling the three-part story, plus a consolidated
figure pack. Every number is pulled from the JSONs; missing inputs are marked,
not invented. **CPU is fine.**

The three-part story: (1) cross-modality DR-grade transfer fails; (2) in-domain
multi-task prediction on OLIVES near-IR (what works); (3) uncertainty behaviour
and its limits.

## 1. Setup & Drive mount

Mount Drive, restore the repo, `cd` in, load the config (reusing the Phase 4a
`zero_shot.yaml` paths). All inputs are saved artefacts on Drive; the output
report goes to the Drive reports dir.

In [ ]:
# Mount Drive and restore the repo.
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

REPO_DIR = "/content/dr-dissertation"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/savita10/dr-dissertation.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} reset --hard origin/main
%cd {REPO_DIR}

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU — set Runtime → Change runtime type → GPU")

In [ ]:
%cd /content/dr-dissertation

In [ ]:
# Load the config (paths reused from Phase 4a).
from pathlib import Path
from src.utils.config import load_config

cfg = load_config('configs/zero_shot.yaml')
print('report_dir :', cfg.report_dir)
print('figures_dir:', cfg.figures_dir)

## 2. STEP 1 — introspection (mandatory, before anything else)

List the report/JSON inputs present in the reports dir and print their top-level
keys, so the consolidation reads the **actual** fields. Any phase not yet run is
reported as MISSING and will appear as a clearly-marked placeholder in the
assembled report (never invented).

In [ ]:
# Report which JSON/markdown inputs are present or missing.
from src.analysis import phase4_report as p4

report_dir = Path(str(cfg.report_dir))
presence = p4.introspect(report_dir)
print('\npresent:', [k for k, v in presence.items() if v])
print('missing:', [k for k, v in presence.items() if not v])

## 3. Assemble the consolidated report

`assemble` reads the four JSONs, pulls every number from them, and writes
`PHASE4_RESULTS.md` (the three-part story + limitations + future work + figure
manifest). It prints the pandoc command to convert to Word later.

In [ ]:
# Assemble and write PHASE4_RESULTS.md to the Drive reports dir.
out_path = p4.assemble(cfg)
print('assembled report:', out_path)

## 4. The consolidated report

Render `PHASE4_RESULTS.md` inline — the dissertation-facing narrative.

In [ ]:
# Display the assembled report.
from IPython.display import Markdown, display

display(Markdown(out_path.read_text(encoding='utf-8')))

## 5. Full figure pack

Display every Phase 1–4 figure currently in the figures dir, in filename order,
so the reader can see the whole visual story alongside the report.

In [ ]:
# Display each figure PNG inline with its caption.
from IPython.display import Image, display

figures_dir = Path(str(cfg.figures_dir))
for png in sorted(figures_dir.glob('*.png')):
    caption = p4.CAPTIONS.get(png.stem, '(caption pending)')
    print(f'\n=== {png.name} — {caption} ===')
    display(Image(filename=str(png)))

## 6. Output manifest

Confirm the consolidated report on Drive and count the figure pack. Note:
`PHASE4_RESULTS.md` can be converted to Word for the dissertation deliverable
with `pandoc PHASE4_RESULTS.md -o PHASE4_RESULTS.docx` (or via LibreOffice).

In [ ]:
# Confirm the artefacts on Drive.
print('consolidated report:', out_path, '(exists:', out_path.exists(), ')')
pngs = sorted(figures_dir.glob('*.png'))
print(f'figure pack: {len(pngs)} PNGs (+ PDF twins) in {figures_dir}')
print('other Phase 3–4 reports present:')
for name in ['phase3a_report.md', 'phase4a_report.md', 'phase4b_report.md',
             'phase4c_report.md', 'PHASE4_RESULTS.md']:
    p = report_dir / name
    print('  ', name, '(exists:', p.exists(), ')')